# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

Todo:
- System prompt for the technical assistant bot
- Function to persist history, make LLM call and stream response
- Tool to maintain LLM course offering platform and course pricing
- Gradio UI
- Accepts audio i/p and returns audio o/p
- Support model switching

In [ ]:
system_prompt = '''
You are a friendly technical assistant with expertise in LLMs. 
When asked for doubts and clarifications on any LLM topic, you should explain in simple terms and ensure that it is understandable for novice.
Answer in the following format:
Give a clean explanation to the question asked
Give a real time relevant example
Explain the concept with an analogy
'''

In [63]:
!uv pip install edge-tts

Using Python 3.12.12 environment at: /Users/madhumithaa/Documents/Repo/llm_engineering/.venv
Resolved 13 packages in 230ms                                        
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/38.88 KiB           
⠙ Preparing packages... (0/2)m------------------ 16.00 KiB/38.88 KiB         
⠙ Preparing packages... (0/2)---------- 32.00 KiB/38.88 KiB         
⠙ Preparing packages... (0/2)---------- 38.88 KiB/38.88 KiB         
⠙ Preparing packages... (0/2)0/2)                                                        
⠙ Preparing packages... (0/2)-------------------     0 B/30.30 KiB           
⠙ Preparing packages... (0/2)--------------- 16.00 KiB/30.30 KiB         
⠙ Preparing packages... (0/2)---------- 30.30 KiB/30.30 KiB         
Prepared 2 packages in 23ms                                                       
Installed 2 packages in 2ms                                 
 + edge

In [73]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import gradio as gr
import sqlite3
import json
import time
import edge_tts
import asyncio

In [81]:
!ollama pull llama3.1:8b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 414 KB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 2.5 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 3.6 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 6.7 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 9.6 MB

In [84]:
CLAUDE_MODEL = "claude-sonnet-4-5"
claude = OpenAI(base_url= os.getenv("ANTHROPIC_BASE_URL"), api_key=os.getenv('ANTHROPIC_API_KEY'))

# OLLAMA_MODEL = "llama3.1:8b"
# ollama = OpenAI(base_url= os.getenv("OLLAMA_BASE_URL"), api_key=os.getenv('OLLAMA_API_KEY'))

LLAMA_MODEL = "llama-3.1-8b-instant"
groq = OpenAI(base_url= os.getenv("GROQ_BASE_URL"), api_key=os.getenv('GROQ_API_KEY'))

DB = "course_prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS course_prices (platform TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [5]:
def get_course_price(platform):
    print(f"DATABASE TOOL CALLED: Getting course price for {platform}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM course_prices WHERE platform = ?', (platform.lower(),))
        result = cursor.fetchone()
        return f"Course price in {platform} is ${result[0]}" if result else "No price data available for this platform"

def set_course_price(platform, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO course_prices (platform, price) VALUES (?, ?) ON CONFLICT(platform) DO UPDATE SET price = ?', (platform.lower(), price, price))
        conn.commit()
        return f"Course price for {platform} has been set to ${price}" or ""
    
get_course_price_function = {
    "name": "get_course_price",
    "description": "Get the price of courses in the platform.",
    "parameters": {
        "type": "object",
        "properties": {
            "platform": {
                "type": "string",
                "description": "The platform in which the customer wants to enroll into a course",
            },
        },
        "required": ["platform"],
        "additionalProperties": False
    }
}

set_course_price_function = {
    "name": "set_course_price",
    "description": "Set the price of a course in the platform.",
    "parameters": {
        "type": "object",
        "properties": {
            "platform": {
                "type": "string",
                "description": "The platform that the customer wants to add or update the course price for",
            },
            "price": {
                "type": "number",
                "description": "The new course price in the platform",
            }
        },
        "required": ["platform", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": get_course_price_function},
    {"type": "function", "function": set_course_price_function}]

In [7]:
course_prices = {"udemy":1500, "coursera": 1800, "greatlearning": 1420, "simplilearn": 5000}
for platform, price in course_prices.items():
    set_course_price(platform, price)

In [6]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        if func_name in globals():
            func = globals()[func_name]
            arguments = json.loads(tool_call.function.arguments)
            result = func(**arguments)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

In [66]:
def talker(message):
    try:
        output_filename = "speech.mp3"
        
        # We pick an open-source high-quality male or female voice
        # "en-US-AndrewNeural" (Male) or "en-US-EmmaNeural" (Female) are excellent.
        voice = "en-US-AndrewNeural" 
        
        # edge-tts is asynchronous, so we run it using the standard event loop
        communicate = edge_tts.Communicate(message, voice)
        asyncio.run(communicate.save(output_filename))
        
        return output_filename
    except Exception as e:
        print(f"Edge-TTS Conversion Error: {e}")
        return None

In [ ]:
def call_model(model, messages):
    if model == "claude":
        return claude.chat.completions.create(model=CLAUDE_MODEL, messages=messages, tools=tools)
    elif model == "groq":
        return groq.chat.completions.create(model=LLAMA_MODEL, messages=messages, tools=tools)
    return

def chat(model, message, history):
    # history = [{"role":h["role"], "content":h["content"]} for h in history]
    # messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    # 1. Convert Gradio's tuple history [[user, assistant], ...] into LLM message format
    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, assistant_msg in history:
        if user_msg:
            messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})
            
    # Add the current new message
    messages.append({"role": "user", "content": message})
    response = call_model(model, messages)

    while response.choices[0].finish_reason=="tool_calls":
        assistant_msg = response.choices[0].message
        responses = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(responses)
        response = call_model(model, messages)
    
    output = response.choices[0].message.content

    words = output.split(" ")
    out = ""
    for i, word in enumerate(words):
        # Add space back to all words except the last one
        out += word + (" " if i < len(words) - 1 else "")
        # Slice off the temporary empty tuple we appended in 'put_message_in_chatbot'
        # and append a fresh list object to force Gradio to re-render the layout
        current_display_history = list(history[:-1]) + [[message, out]]
        
        yield current_display_history, None
        time.sleep(0.03)

    # Generate the audio once the text is fully rendered
    voice = talker(output)

    # Yield the finalized history block along with the audio file
    history[-1] = [message, output]
    yield history, voice


In [90]:
def put_message_in_chatbot(message, history):
    # return "", history + [{"role":"user", "content":message}]
    # In tuple format, we append a new empty pair: [user_message, None]
    # msg = {"role":"user", "content":message}
    return "", history + [[message, None]]

with gr.Blocks() as ui:
    with gr.Row():
        model_selector = gr.Dropdown(label="Select Model", choices=["claude", "groq"], value="claude")
    with gr.Row():
        chatbot = gr.Chatbot(label="Chat with LLM")
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message_input = gr.Textbox(label="Enter your message here", placeholder="Type your message and press Enter")

    # Hooking up events to callbacks

    message_input.submit(put_message_in_chatbot, inputs=[message_input, chatbot], outputs=[message_input, chatbot]).then(
        chat, inputs=[model_selector, message_input, chatbot], outputs=[chatbot, audio_output]
    )

ui.launch()

/var/folders/3x/bznhcm4154lcwqy_5pwtr4t00000gn/T/ipykernel_29515/3960745049.py:11: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat with LLM")


* Running on local URL:  http://127.0.0.1:7923
* To create a public link, set `share=True` in `launch()`.
